In [1]:
import mailbox
import pandas as pd
import os
import html2text

In [5]:
#read files
mbox_files = ['../DataSet/raw/phishing-2022.mbox',
              '../DataSet/raw/phishing-2023.mbox',
              '../DataSet/raw/phishing-2024.mbox']

#function for extract body
def extract_body(msg):
    body = []
    if msg.is_multipart():
        for part in msg.walk():
            ctype = part.get_content_type()
            payload = part.get_payload(decode = True)

            if payload is None:
                continue

            if ctype == 'text/plain':
                text = payload.decode(errors = 'ignore').strip()
                if text:
                    body.append(text)
                break
            elif ctype == 'text/html':
                html = payload.decode(errors = 'ignore')
                text = html2text.html2text(html).strip()
                if text:
                    body.append(text)
    else:
        payload = msg.get_payload(decode = True)
        if payload:
            ctype = msg.get_content_type()
            if ctype == 'text/plain':
                body.append(payload.decode(errors = 'ignore').strip())
            elif ctype == 'text/html':
                html = payload.decode(errors = 'ignore')
                body.append(html2text.html2text(html).strip())

    full_body = '\n\n'.join(body).strip()
    return full_body

In [7]:
#Extract Content, Add Dummy label as Phishing Email
data_all = []

for mbox_file in mbox_files:
    mbox = mailbox.mbox(mbox_file)
    count_raw = 0
    count_valid = 0

    for msg in mbox:
        count_raw += 1
        body = extract_body(msg)

        #remove short mail
        if len(body) > 30:
            data_all.append({
                'Email Text': body,
                'Email Type': 'Phishing Email'
            })
            count_valid += 1
    print(f"Valid Mail Ratio: {count_valid} / {count_raw}")

#combine files and remove duplicate
df_mbox_new = pd.DataFrame(data_all)
count_valid = len(df_mbox_new)
df_mbox_new.drop_duplicates(subset = ['Email Text'], inplace = True)
print(f"Case Mail Ratio: {len(df_mbox_new)} / {count_valid}")

#output file
output_csv = 'combined_mbox_phishing_clean_2text.csv'
df_mbox_new.to_csv(output_csv, index = False, encoding = 'utf-8')
df_mbox_new.head(20)

Valid Mail Ratio: 242 / 247
Valid Mail Ratio: 415 / 419
Valid Mail Ratio: 396 / 403
Case Mail Ratio: 966 / 1053


,Email Text,Email Type
0,Microsoft Failure Delivery Notice.\n User: jo...,Phishing Email
3,Microsoft Failure Delivery Notice.\n User: jo...,Phishing Email
4,Microsoft Failure Delivery Notice.\n User: jo...,Phishing Email
5,New Voice Message\n New Caller left a message...,Phishing Email
7,"February 02, 2022\t \t \n \t \t \n \t \t \...",Phishing Email
9,Microsoft Failure Delivery Notice.\n User: jo...,Phishing Email
10,| | | | \n--- \n**** \n**** \n--- \n ...,Phishing Email
11,您好!\n您的邮箱账号：jose@monkey.org，将在以下时间终止。\n\n12:15...,Phishing Email
12,あなたのアカウントは停止されました\r\n\r\n \r\n新しいデバイスからアカウントサー...,Phishing Email
13,"Hello\nYou received a file\n 1 item, 3MB in t...",Phishing Email
